In [ ]:
import os
import glob
import numpy as np
from PIL import Image
from tqdm import tqdm

def slice_image(img_path, save_dir, tile_size=512, overlap=0):
    """
    Slices a large image into a grid of smaller tiles.
    """
    img = Image.open(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    w, h = img.size
    
    # Calculate number of tiles
    # We use ceil to ensure we get the edges, padding if necessary
    x_steps = int(np.ceil(w / tile_size))
    y_steps = int(np.ceil(h / tile_size))

    for i in range(x_steps):
        for j in range(y_steps):
            # Define the window
            left = i * tile_size
            upper = j * tile_size
            right = min(left + tile_size, w)
            lower = min(upper + tile_size, h)
            
            # Crop
            tile = img.crop((left, upper, right, lower))
            
            # If tile is smaller than target (at edges), pad it with zeros
            if tile.size != (tile_size, tile_size):
                new_tile = Image.new(img.mode, (tile_size, tile_size), 0)
                new_tile.paste(tile, (0, 0))
                tile = new_tile
            
            # Save: e.g., "flood_01_amp_row0_col1.png"
            save_path = os.path.join(save_dir, f"{img_name}_r{j}_c{i}.png")
            tile.save(save_path)

def process_folder(input_folder, output_folder, tile_size=512):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    # Find all files
    files = glob.glob(os.path.join(input_folder, "*.png"))
    print(f"Slicing {len(files)} files from {input_folder}...")

    for f in tqdm(files):
        slice_image(f, output_folder, tile_size=tile_size)

if __name__ == "__main__":
    # --- CONFIGURATION ---
    # Adjust tile_size based on your GPU memory (512, 1024, etc.)
    TILE_SIZE = 512 
    
    # Input folders (where your huge 9k images are)
    train_input = "dataset_pngs/train"
    val_input = "dataset_pngs/val"
    
    # Output folders (where the chips will go)
    train_output = "dataset_tiled/train"
    val_output = "dataset_tiled/val"
    # ---------------------
    
    process_folder(train_input, train_output, tile_size=TILE_SIZE)
    process_folder(val_input, val_output, tile_size=TILE_SIZE)
    
    print("Done! Now point your U-Net to 'dataset_tiled'.")

Slicing 12 files from dataset_pngs/train...


100%|██████████| 12/12 [00:21<00:00,  1.81s/it]


Slicing 3 files from dataset_pngs/val...


100%|██████████| 3/3 [00:05<00:00,  1.85s/it]

Done! Now point your U-Net to 'dataset_tiled'.
